In [1]:
import numpy as np
import utils
import os
from os.path import exists

In [2]:
logdm2_list = np.asarray(np.ndarray.tolist(np.arange(-2.10, 2.10, 0.1))+ [2.15])
dm2_list = np.power(10, logdm2_list)  
dm2_list = dm2_list[np.logical_and(dm2_list>.1, dm2_list<50)]
dm2_list = np.append(dm2_list, 50.0)

log_Um4sq_list = np.asarray(np.ndarray.tolist(np.arange(-3., 0., 0.1)))  
Um4sq_list = np.power(10, log_Um4sq_list)
Um4sq_list = Um4sq_list[Um4sq_list<0.50]
Um4sq_list = np.append(Um4sq_list, 0.50)

log_Ut4sq_list = np.asarray(np.ndarray.tolist(np.arange(-3., 0., 0.2)))   
Ut4sq_list = np.power(10, log_Ut4sq_list)  
Ut4sq_list = np.insert(Ut4sq_list, 0, 0.)
Ut4sq_list = Ut4sq_list[Ut4sq_list<0.50]
Ut4sq_list = np.append(Ut4sq_list, 0.50)

d24_list = [np.pi]

In [3]:
Ut4sq_list

array([0.        , 0.001     , 0.00158489, 0.00251189, 0.00398107,
       0.00630957, 0.01      , 0.01584893, 0.02511886, 0.03981072,
       0.06309573, 0.1       , 0.15848932, 0.25118864, 0.39810717,
       0.5       ])

In [4]:
fit_dict = {
    'use_fastMC' : 'True',
    'Emax' : '9976',
    'sim' : 'Platinum_Split_Plus_Tau',
    'max_sim' : '-1', 
    'fast_scaling' : '0.50',
    'fastmc_dir' : '/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/FastMC/',
#     'realization_file' : realization_file,
    'years' : '7.634',
    'osg' : 'True',
    'crSlopeWidth': '0.25'
}

In [5]:
def print_vars(f, jobname, var_name, val):
    f.write('VARS\t' + jobname.strip() + '\t' + var_name+'=\"' + val+ "\"\n")
    
inject_point = "0_0.000000_0.000000_0.000000_0.000000_0.000000_0.000000"

dir_base = '/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/BrazilPlot/AIRS_mceq121_2011_sib_HG_E3/'+inject_point



In [6]:
# for i in range(0,1999):
#     for Ut4sq in Ut4sq_list:
#         dirname = dir_base + "/fits/Realization_{i}/Ut4sq_{Ut4sq:.6f}".format(i=i, Ut4sq=Ut4sq)
#         os.makedirs(dirname, exist_ok=True)

In [7]:
def getR_seq(dirbase, point, Ut4sq, R_MIN, R_MAX):
    seq = []
    for i in range(R_MIN, R_MAX):
        filename = dirbase + "/fits/Realization_{i}/Ut4sq_{Ut4sq:.6f}/{point}_Realization_{i}.json".format(i=i, Ut4sq=Ut4sq, point = point)
        if not exists(filename):
            seq.append(i)

    if not seq:
        return None
    
    seq_str = ' '.join([str(s) for s in seq])
    
    return "'"+seq_str+"'"

def make_fit_dag(filename, jobname, fluxdir, point, outdir, Ut4sq, seq, n_seeds, osg = "True"):
    if osg == "True":
        filename.write('JOB\t' + jobname + '\t/home/adiaz/IC-MEOWS-th34/BrazilPlot/do_fit_OSG.submit\n')
    else:
        filename.write('JOB\t' + jobname + '\t/home/adiaz/IC-MEOWS-th34/BrazilPlot/do_fit.submit\n')
    print_vars(filename, jobname, 'point', point)
    print_vars(filename, jobname, 'fluxdir', fluxdir)
    print_vars(filename, jobname, 'outdir', outdir)
    print_vars(filename, jobname, 'Ut4sq', "{:.6f}".format(Ut4sq))
    print_vars(filename, jobname, 'seq', seq)
    print_vars(filename, jobname, 'n_seeds', n_seeds)
    for key, value in fit_dict.items():
        print_vars(filename, jobname, key, value)
#     filename.write('Retry ' + jobname + " 2\n")



In [8]:
def make_dag(low, high, step, n_seeds = '-1', retry = 0):
    i=0
    edges = np.arange(low,high,step)
    for r in range(len(edges)-1):
        R_MIN = edges[r]
        R_MAX = edges[r+1]
        print(R_MIN,R_MAX)
#         dag_filename = "./indivigies/{}.{}_{}.dag".format(inject_point, R_MIN, R_MAX)
        dag_filename = "./{}.{}_{}.dag".format(inject_point, R_MIN, R_MAX)
        osg = fit_dict['osg']
        with open(dag_filename, 'w') as f:
            n_fits=0
            Ue4sq = 0.
            del14 = 0.
            del24 = d24_list[0]
            for Ut4sq in Ut4sq_list:
                print(Ut4sq)
                fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'

                if Ut4sq == 0.:
                    point = utils.get_point_string()
                    seq = getR_seq(dir_base, point, Ut4sq, R_MIN, R_MAX)
                    if seq!=None:
                        make_fit_dag(f, str(i), fluxdir, point, dir_base, Ut4sq, seq, n_seeds, osg)
                        n_fits += len(seq.split(' '))
                        if retry > 1:
                            f.write('Retry ' + str(i) + " {}\n".format(retry))
                        i+=1

                for Um4sq in Um4sq_list:
                    for dm2 in dm2_list:

                        th14, th24, th34 = utils.Usq_to_thetas(Ue4sq, Um4sq, Ut4sq)
                        point = utils.get_point_string(index = 0, dm2 = dm2, th14 = th14, th24 = th24, th34 = th34, d14 = del14, d24 = del24)
                        seq = getR_seq(dir_base, point, Ut4sq, R_MIN, R_MAX)
                        if seq!=None:
                            make_fit_dag(f, str(i), fluxdir, point, dir_base, Ut4sq, seq, n_seeds, osg)
                            if retry > 1:
                                f.write('Retry ' + str(i) + " {}\n".format(retry))
                            n_fits += len(seq.split(' '))
                            i += 1

        print(n_fits)


In [ ]:
make_dag(0,101,10)

In [ ]:
make_dag(100,201,10)

In [ ]:
make_dag(200,301,10)